In [ ]:
!pip install ultralytics -q

In [ ]:
import yaml
from pathlib import Path

BASE = Path("/kaggle/input/datasets/sergeisemenovdev/oranges-detection-yolov8/oranges-detection-yolov8-dataset")

# Создаём исправленный data.yaml с абсолютными путями
data = {
    "train": str(BASE / "train" / "images"),
    "val":   str(BASE / "valid" / "images"),
    "test":  str(BASE / "test"  / "images"),
    "nc": 2,
    "names": ["Fresh Orange", "Rotten Orange"],
}

yaml_path = Path("/kaggle/working/data.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(data, f, allow_unicode=True)

print("data.yaml создан:")
print(yaml_path.read_text())

# Проверка количества файлов
for split in ("train", "valid", "test"):
    imgs = list((BASE / split / "images").glob("*.png")) + \
           list((BASE / split / "images").glob("*.jpg"))
    lbls = list((BASE / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} изображений, {len(lbls)} labels")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

results = model.train(
    data="/kaggle/working/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/kaggle/working/runs",
    name="oranges_v2_s",
    patience=10,
    save=True,
    plots=True,
)

In [ ]:
best_model = YOLO("/kaggle/working/runs/oranges_v2_s/weights/best.pt")

metrics = best_model.val(
    data="/kaggle/working/data.yaml",
    split="test",
    device=0,
)

print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
import random
from pathlib import Path

test_images = list((BASE / "test" / "images").glob("*.png")) + \
              list((BASE / "test" / "images").glob("*.jpg"))

sample = random.sample(test_images, min(5, len(test_images)))

best_model.predict(
    source=sample,
    device=0,
    save=True,
    project="/kaggle/working/runs",
    name="oranges_v1_predict",
    conf=0.25,
)

print("Предсказания сохранены в /kaggle/working/runs/oranges_v1_predict/")

In [ ]:
from IPython.display import FileLink

FileLink("/kaggle/working/runs/oranges_v2_s/weights/best.pt")

In [ ]:
import shutil
shutil.copy(
    "/kaggle/working/runs/oranges_v2_s/weights/best.pt",
    "/kaggle/working/best.pt"
)
print("Готово. Теперь скачай файл через панель Output.")